In [1]:
import os, sys
REPO = "https://github.com/salsyaniditiani-commits/simuteh--2330511061-.git"

if not os.path.exists("/content/simuteh"):
    !git clone {REPO} /content/simuteh
os.chdir("/content/simuteh")

for f in ["notebook", "hasil", "laporan"]:
    os.makedirs(f, exist_ok=True)

if not os.path.exists("data/sintetis/citra"):
    !python src/buat_dataset_sintetis.py

os.chdir("/content/simuteh/notebook")
sys.path.append("../src")

print("Folder kerja:", os.getcwd())
print("Jumlah citra:", len(os.listdir("../data/sintetis/citra")))
print("Isi src:", os.listdir("../src"))

Cloning into '/content/simuteh'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 111 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 13.43 MiB | 20.87 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Selesai: 60 citra tersimpan di data/sintetis
Folder kerja: /content/simuteh/notebook
Jumlah citra: 60
Isi src: ['buat_dataset_sintetis.py', 'utils_cv.py']


Kode 4.1 Verifikasi konvolusi manual

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

from utils_cv import baca, statistik, tampil, kontras_bercak

# Membaca citra dan mask
nama = "teh_041.png"

bersih = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nama}"), cv2.COLOR_BGR2GRAY)
md = cv2.imread(f"../data/sintetis/mask_daun/{nama}", 0)
mb = cv2.imread(f"../data/sintetis/mask_bercak/{nama}", 0)

# Fungsi konvolusi manual
def konvolusi(img, k):  # fungsi dari slide P4
    kh, kw = k.shape
    ph, pw = kh // 2, kw // 2
    pad = np.pad(img.astype(float), ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros(img.shape)

    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y, x] = np.sum(pad[y:y+kh, x:x+kw] * k)

    return np.clip(np.round(out), 0, 255).astype(np.uint8)


# Fungsi konvolusi sejati
def konvolusi_sejati(img, k):  # kernel dibalik 180 derajat dahulu
    return konvolusi(img, np.flip(k))


# Pengujian beberapa kernel
for nm, k in {
    "rerata": np.ones((3, 3)) / 9,
    "asimetris": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float)
}.items():

    t0 = time.perf_counter()
    a = konvolusi(bersih, k)
    t_manual = time.perf_counter() - t0

    t0 = time.perf_counter()
    b = cv2.filter2D(bersih, -1, k, borderType=cv2.BORDER_REPLICATE)
    t_cv = time.perf_counter() - t0

    beda = np.abs(a.astype(int) - b.astype(int)).max()

    c = konvolusi_sejati(bersih, k)
    beda2 = np.abs(c.astype(int) - b.astype(int)).max()

    print(f"{nm}: selisih manual vs filter2D={beda}, "
          f"sejati vs filter2D={beda2}, "
          f"waktu manual={t_manual:.3f} s, filter2D={t_cv:.5f} s")

rerata: selisih manual vs filter2D=0, sejati vs filter2D=0, waktu manual=2.480 s, filter2D=0.00938 s
asimetris: selisih manual vs filter2D=0, sejati vs filter2D=255, waktu manual=2.475 s, filter2D=0.00041 s


Kode 4.2 Tabel penapis x derau x PSNR x waktu x KB

In [4]:
import pandas as pd
rng = np.random.default_rng(42)

def derau_gauss(img, sd):
    return np.clip(img + rng.normal(0, sd, img.shape), 0, 255).astype(np.uint8)

def derau_sp(img, p):                  # p = proporsi piksel rusak
    out = img.copy(); m = rng.random(img.shape)
    out[m < p / 2] = 0; out[m > 1 - p / 2] = 255
    return out

def ukur_waktu(f, x, ulang=20):        # rerata beberapa kali, dalam ms
    t = []
    for _ in range(ulang):
        t0 = time.perf_counter(); f(x); t.append(time.perf_counter() - t0)
    return 1000 * np.mean(t)

derau = {"gauss_10": derau_gauss(bersih, 10), "gauss_20": derau_gauss(bersih,
20),
         "gauss_40": derau_gauss(bersih, 40), "sp_4%": derau_sp(bersih, 0.04),
         "sp_10%": derau_sp(bersih, 0.10)}
penapis = {"rerata_3": lambda x: cv2.blur(x, (3, 3)),
           "rerata_5": lambda x: cv2.blur(x, (5, 5)),
           "gauss_5": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0),
           "median_3": lambda x: cv2.medianBlur(x, 3),
           "median_5": lambda x: cv2.medianBlur(x, 5),
           "median_9": lambda x: cv2.medianBlur(x, 9)}
baris = []
for dn, d in derau.items():
    baris.append([dn, "tanpa", cv2.PSNR(bersih, d), 0.0,
                  kontras_bercak(d, md, mb)])
    for pn, f in penapis.items():
        out = f(d)
        baris.append([dn, pn, cv2.PSNR(bersih, out), ukur_waktu(f, d),
                      kontras_bercak(out, md, mb)])
tabel = pd.DataFrame(baris, columns=["derau", "penapis", "PSNR_dB",
                                     "waktu_ms", "KB"]).round(2)
print(tabel)
tabel.to_csv("../hasil/b4_tabel_penapis.csv", index=False)

       derau   penapis  PSNR_dB  waktu_ms    KB
0   gauss_10     tanpa    28.11      0.00  3.64
1   gauss_10  rerata_3    36.97      0.98  6.12
2   gauss_10  rerata_5    38.11      1.19  6.33
3   gauss_10   gauss_5    37.86      0.99  6.30
4   gauss_10  median_3    35.53      0.82  6.00
5   gauss_10  median_5    38.58      4.67  6.89
6   gauss_10  median_9    38.95     59.37  7.52
7   gauss_20     tanpa    22.10      0.00  2.04
8   gauss_20  rerata_3    31.45      0.43  4.72
9   gauss_20  rerata_5    34.70      0.57  5.62
10  gauss_20   gauss_5    32.61      1.35  5.08
11  gauss_20  median_3    29.75      0.93  4.20
12  gauss_20  median_5    33.54      6.06  5.48
13  gauss_20  median_9    35.79     41.96  6.10
14  gauss_40     tanpa    16.33      0.00  1.02
15  gauss_40  rerata_3    25.80      0.14  2.86
16  gauss_40  rerata_5    29.83      0.17  4.06
17  gauss_40   gauss_5    27.03      0.30  3.21
18  gauss_40  median_3    23.83      0.11  2.35
19  gauss_40  median_5    27.88      0.9

Kode 4.3 Pengaruh ukuran kernel median pada bercak kecil

In [5]:
def selisih_rerata(kanal, mask_daun, mask_bercak):
    """dI = |rerata bercak - rerata daun sehat|, dalam satuan intensitas."""
    k = kanal.astype(float)
    sehat = (mask_daun > 0) & (mask_bercak == 0)
    return abs(k[mask_bercak > 0].mean() - k[sehat].mean())

label = pd.read_csv("../data/sintetis/label.csv")
baris = []
for nm in label[label.kelas == "ringan"].nama_file:
    g = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nm}"), cv2.COLOR_BGR2GRAY)
    md2 = cv2.imread(f"../data/sintetis/mask_daun/{nm}", 0)
    mb2 = cv2.imread(f"../data/sintetis/mask_bercak/{nm}", 0)
    for k in [1, 3, 5, 9, 15]:          # k = 1 berarti tanpa penapis
        out = g if k == 1 else cv2.medianBlur(g, k)
        baris.append([nm, k, selisih_rerata(out, md2, mb2),
                      kontras_bercak(out, md2, mb2)])
hasil_median = pd.DataFrame(baris, columns=["nama", "kernel", "dI", "KB"])
print(hasil_median.groupby("kernel")[["dI", "KB"]].mean().round(2))

           dI    KB
kernel             
1       25.84  3.91
3       26.43  4.06
5       25.78  4.02
9       20.06  3.62
15       8.05  2.13


Kode 4.4 Pengaruh urutan penajaman

In [7]:

def unsharp(x, k=1.0):
    blur = cv2.GaussianBlur(x, (5, 5), 1.0)
    return cv2.addWeighted(x, 1 + k, blur, -k, 0)

noisy = derau["gauss_20"]
latar = md == 0                         # area latar yang seharusnya rata
for nm, redam in {"Gaussian": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0),
                  "Median": lambda x: cv2.medianBlur(x, 5)}.items():
    A = redam(unsharp(noisy))           # tajamkan dulu, lalu redam
    B = unsharp(redam(noisy))           # redam dulu, lalu tajamkan
    print(f"{nm}: tajam->redam PSNR={cv2.PSNR(bersih, A):.2f} "
          f"sigma_latar={A[latar].std():.1f} | redam->tajam "
          f"PSNR={cv2.PSNR(bersih, B):.2f} sigma_latar={B[latar].std():.1f}")

Gaussian: tajam->redam PSNR=30.34 sigma_latar=8.0 | redam->tajam PSNR=30.20 sigma_latar=8.2
Median: tajam->redam PSNR=30.20 sigma_latar=8.1 | redam->tajam PSNR=31.16 sigma_latar=7.2
